In [ ]:

# CELL 1: SETUP AND DATASET EXTRACTION

!pip -q install -U ultralytics pyyaml

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import zipfile
import shutil
import yaml
import os

# Find the ZIP anywhere inside My Drive
drive_root = Path("/content/drive/MyDrive")
zip_matches = list(drive_root.rglob("step7b4_yolo_dataset.zip"))

if not zip_matches:
    raise FileNotFoundError(
        "step7b4_yolo_dataset.zip was not found in My Drive. "
        "Upload it to Google Drive and rerun this cell."
    )

if len(zip_matches) > 1:
    print("Multiple ZIP files found:")
    for p in zip_matches:
        print(p)
    raise RuntimeError(
        "Please keep/select one intended ZIP file and rerun this cell."
    )

ZIP_PATH = zip_matches[0]
EXTRACT_DIR = Path("/content/step7b4_yolo_dataset")

print("ZIP found:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

# Re-extract into temporary Colab storage.
# The original ZIP on Drive is not changed.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad_member = z.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
    z.extractall(EXTRACT_DIR)

yaml_files = list(EXTRACT_DIR.rglob("data.yaml"))

if len(yaml_files) != 1:
    print("Found data.yaml files:", yaml_files)
    raise RuntimeError(
        f"Expected exactly one data.yaml, found {len(yaml_files)}. "
        "Inspect the extracted folder before proceeding."
    )

DATA_YAML = yaml_files[0]

print("\nExtraction complete.")
print("Dataset configuration:", DATA_YAML)
print("\n--- Original data.yaml ---")
print(DATA_YAML.read_text())
print("\n--- Top-level extracted contents ---")
for p in sorted(EXTRACT_DIR.iterdir()):
    print(("DIR " if p.is_dir() else "FILE"), p.name)


In [ ]:

# CELL 2: VERIFY DATASET BEFORE TRAINING

from pathlib import Path
import yaml
import re
import shutil

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

DATA_ROOT = DATA_YAML.parent

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in
        {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ]

def find_split_images(split):
    candidates = [
        DATA_ROOT / "images" / split,
        DATA_ROOT / split / "images",
        DATA_ROOT / split,
    ]

    for p in candidates:
        if p.is_dir() and image_files(p):
            return p.resolve()

    # Fallback for a ZIP with an extra nested directory
    matches = [
        p for p in DATA_ROOT.rglob("*")
        if p.is_dir()
        and p.name == split
        and image_files(p)
        and ("images" in p.parts or p.parent.name == "images")
    ]

    if len(matches) == 1:
        return matches[0].resolve()

    raise FileNotFoundError(
        f"Could not uniquely locate the {split} image folder. "
        f"Candidates: {matches}"
    )

def infer_label_folder(image_dir, split):
    image_dir = Path(image_dir)

    if image_dir.parent.name == "images":
        root = image_dir.parent.parent
        candidates = [root / "labels" / split]

    elif image_dir.name == "images":
        root = image_dir.parent
        candidates = [root / "labels" / split]

    else:
        candidates = [
            image_dir.parent / "labels",
            image_dir.parent / "labels" / split,
            DATA_ROOT / "labels" / split,
        ]

    for p in candidates:
        if p.is_dir():
            return p.resolve()

    raise FileNotFoundError(
        f"Cannot find the {split} labels folder for {image_dir}. "
        "Do not start training until the label structure is verified."
    )

split_images = {}
split_labels = {}

for split in ["train", "val", "test"]:
    split_images[split] = find_split_images(split)
    split_labels[split] = infer_label_folder(
        split_images[split], split
    )

    images = image_files(split_images[split])
    labels = list(split_labels[split].glob("*.txt"))

    print(
        f"{split}: {len(images)} images | "
        f"{len(labels)} label TXT files"
    )
    print("  Images:", split_images[split])
    print("  Labels:", split_labels[split])

    if not images:
        raise RuntimeError(f"No images found in {split}.")

# Verify class mapping.
names = cfg.get("names")

if isinstance(names, list):
    class_names = {i: str(n) for i, n in enumerate(names)}
elif isinstance(names, dict):
    class_names = {int(k): str(v) for k, v in names.items()}
else:
    raise RuntimeError(
        "data.yaml has no usable 'names' field. "
        "Verify the original class mapping before training."
    )

print("\nClass mapping:", class_names)

if set(class_names.keys()) != {0, 1}:
    raise RuntimeError(
        "Expected exactly two classes with IDs 0 and 1. "
        f"Found {class_names}. Do not train until corrected."
    )

# Expected project mapping:
# 0 = Nutrition Deficiency
# 1 = Weed / Weedy Rice
def normalize_name(s):
    return re.sub(r"[^a-z0-9]+", "", s.lower())

name0 = normalize_name(class_names[0])
name1 = normalize_name(class_names[1])

if not (
    ("nutrition" in name0 or "nutrient" in name0)
    and ("weed" in name1 or "weedy" in name1)
):
    raise RuntimeError(
        "Class names/order do not match the expected project mapping. "
        "Expected class 0 = Nutrition Deficiency and class 1 = "
        "Weed / Weedy Rice. Actual mapping: "
        f"{class_names}. Verify the source annotations before training."
    )

# Check every polygon label:
# class_id x1 y1 x2 y2 x3 y3 ...
bad_labels = []
class_counts = {0: 0, 1: 0}
valid_label_lines = 0

for split, label_dir in split_labels.items():
    for label_file in label_dir.glob("*.txt"):
        for line_no, line in enumerate(
            label_file.read_text().splitlines(), start=1
        ):
            if not line.strip():
                continue

            parts = line.split()

            try:
                values = [float(v) for v in parts]
                class_id = int(values[0])

                if values[0] != class_id or class_id not in (0, 1):
                    raise ValueError("Invalid class ID")

                coords = values[1:]

                if len(coords) < 6 or len(coords) % 2 != 0:
                    raise ValueError("Polygon needs at least 3 points")

                if not all(0.0 <= v <= 1.0 for v in coords):
                    raise ValueError("Coordinates outside [0, 1]")

                class_counts[class_id] += 1
                valid_label_lines += 1

            except Exception as e:
                bad_labels.append(
                    f"{label_file}:{line_no}: {e}"
                )

if bad_labels:
    print("\nInvalid annotations (first 20):")
    print("\n".join(bad_labels[:20]))
    raise RuntimeError(
        f"Found {len(bad_labels)} invalid polygon lines. "
        "Training has been stopped."
    )

print("\nValid polygon annotations:", valid_label_lines)
print("Polygon counts by class:", class_counts)

if valid_label_lines == 0:
    raise RuntimeError("No valid polygon annotations were found.")

# Point data.yaml directly to the extracted image folders.
# Preserve the existing, verified class mapping.
cfg["path"] = str(DATA_ROOT.resolve())
cfg["train"] = str(split_images["train"])
cfg["val"] = str(split_images["val"])
cfg["test"] = str(split_images["test"])

VERIFIED_YAML = Path("/content/step7b4_data_verified.yaml")

with open(VERIFIED_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("\nVerified data.yaml saved to:", VERIFIED_YAML)
print(VERIFIED_YAML.read_text())
print("\nDATASET CHECKS PASSED.")


In [ ]:

# CELL 3: YOLO26N-SEG TRAINING WITH RESUME SUPPORT

import torch
import shutil
from pathlib import Path
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable a GPU in Colab Runtime settings "
        "before starting training."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# All persistent training outputs are stored on Google Drive.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/CDAC_YOLO_Training"
)
RUN_NAME = "yolo26n_seg_agriculture"
RUN_DIR = PROJECT_DIR / RUN_NAME
WEIGHTS_DIR = RUN_DIR / "weights"

LAST_PT = WEIGHTS_DIR / "last.pt"
BEST_PT = WEIGHTS_DIR / "best.pt"
LATEST_PT = WEIGHTS_DIR / "latest.pt"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Called after Ultralytics saves a model checkpoint.
# last.pt is the native resume checkpoint.
# latest.pt is an additional copy for convenient access.
def sync_checkpoints(trainer):
    last_path = Path(trainer.last)
    best_path = Path(trainer.best)

    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

    if last_path.exists():
        shutil.copy2(last_path, LATEST_PT)

    if best_path.exists():
        # best.pt is saved by Ultralytics itself.
        # This check confirms it exists in the persistent run folder.
        print("Best checkpoint available:", best_path)

# If a prior run exists, resume it rather than restarting.
if LAST_PT.exists():
    print("\nExisting last.pt found:", LAST_PT)
    print("Resuming training from the saved checkpoint...")

    model = YOLO(str(LAST_PT))
    model.add_callback("on_model_save", sync_checkpoints)

    # Resume restores the saved training state and continues training.
    model.train(resume=True)

else:
    print("\nNo previous checkpoint found. Starting fresh training.")

    model = YOLO("yolo26n-seg.pt")
    model.add_callback("on_model_save", sync_checkpoints)

    model.train(
        data=str(VERIFIED_YAML),
        epochs=50,
        batch=32,
        imgsz=640,
        device=0,
        workers=2,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        optimizer="auto",
        amp=True,
        cache=False,
        save=True,
        save_period=-1,
        patience=50,
        plots=True,
        seed=42,
        deterministic=True,
        verbose=True,
    )

print("\nTraining command finished.")
print("Last checkpoint:", LAST_PT, LAST_PT.exists())
print("Latest checkpoint:", LATEST_PT, LATEST_PT.exists())
print("Best checkpoint:", BEST_PT, BEST_PT.exists())


In [ ]:

from pathlib import Path
import yaml

run_dir = Path(
    "/content/drive/MyDrive/CDAC_YOLO_Training/"
    "yolo26n_seg_agriculture"
)

args_file = run_dir / "args.yaml"

if args_file.exists():
    args = yaml.safe_load(args_file.read_text())
    for key in [
        "model", "epochs", "batch", "imgsz", "optimizer",
        "lr0", "lrf", "weight_decay", "patience",
        "amp", "mosaic", "mixup", "copy_paste",
        "close_mosaic", "flipud", "fliplr"
    ]:
        print(f"{key}: {args.get(key, 'not recorded')}")
else:
    print("args.yaml not found yet. Check the run folder after it is created.")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

# Search the expected training folder first
training_folder = drive_root / "CDAC_YOLO_Training"

matches = list(training_folder.rglob("best.pt")) if training_folder.exists() else []

if matches:
    print("Found best.pt:")
    for path in matches:
        print(path)
else:
    print("best.pt not found in CDAC_YOLO_Training.")
    print("Searching MyDrive for saved checkpoints...")
    matches = list(drive_root.rglob("best.pt"))
    for path in matches[:30]:
        print(path)

    if not matches:
        print("No best.pt found in MyDrive. Check that the correct Google account is mounted.")


In [ ]:

from pathlib import Path
from ultralytics import YOLO
import torch

# Load our saved best model
BASE = Path("/content/drive/MyDrive/CDAC_YOLO_Training")
RUN = BASE / "yolo26n_seg_agriculture"
BEST = RUN / "weights" / "best.pt"

# Recreate the verified dataset configuration if necessary
DATA_YAML = Path("/content/step7b4_data_verified.yaml")

assert BEST.exists(), f"Missing checkpoint: {BEST}"
assert DATA_YAML.exists(), (
    "Verified dataset YAML is missing. "
    "Rerun dataset setup and validation cells first."
)
assert torch.cuda.is_available(), "Enable a GPU runtime first."

model = YOLO(str(BEST))

# Independent evaluation on held-out test data
results = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,
    project=str(BASE),
    name="yolo26n_seg_test_evaluation",
    exist_ok=True,
)

print("\n========== TEST SET RESULTS ==========")
for metric, value in results.results_dict.items():
    print(f"{metric}: {value}")

print("\nEvaluation completed.")
print("Best model:", BEST)
print("Results folder:", BASE / "yolo26n_seg_test_evaluation")


In [ ]:

from pathlib import Path
import yaml
from ultralytics import YOLO

BASE = Path("/content/drive/MyDrive/CDAC_YOLO_Training")
BEST = BASE / "yolo26n_seg_agriculture" / "weights" / "best.pt"
DATA_YAML = Path("/content/step7b4_data_verified.yaml")

# Read the dataset configuration
with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

dataset_root = Path(config.get("path", DATA_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = DATA_YAML.parent / dataset_root

test_entry = Path(config["test"])
test_dir = test_entry if test_entry.is_absolute() else dataset_root / test_entry

assert BEST.exists(), f"Model not found: {BEST}"
assert test_dir.exists(), f"Test images not found: {test_dir}"

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
test_images = sorted(
    p for p in test_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

assert test_images, f"No test images found in {test_dir}"

print("Test image directory:", test_dir)
print("Number of test images:", len(test_images))

model = YOLO(str(BEST))

output_dir = BASE / "test_visual_predictions"

# Visualize a small sample first
results = model.predict(
    source=[str(p) for p in test_images[:12]],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=str(BASE),
    name="test_visual_predictions",
    exist_ok=True
)

print("\nPredictions saved to:", output_dir)
print("Review these images and check whether both classes are segmented correctly.")


In [ ]:

# ============================================================
# CDAC AGRICULTURE SEGMENTATION — COMPLETE EVALUATION REPORT
# Model: YOLO26 Nano Segmentation
# ============================================================

from pathlib import Path
import os
import csv
import yaml
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# -------------------- 1. CONFIGURATION --------------------

BASE = Path("/content/drive/MyDrive/CDAC_YOLO_Training")
BEST = BASE / "yolo26n_seg_agriculture" / "weights" / "best.pt"
DATA_YAML = Path("/content/step7b4_data_verified.yaml")

IMG_SIZE = 640
BATCH_SIZE = 16
CONF_THRESHOLD = 0.25

CLASS_NAMES = {
    0: "Nutrition_Deficiency",
    1: "Weed"
}

assert BEST.exists(), f"Model checkpoint not found: {BEST}"
assert DATA_YAML.exists(), (
    f"Dataset YAML not found: {DATA_YAML}. "
    "Rerun the dataset extraction and verification cells."
)
assert torch.cuda.is_available(), (
    "GPU not available. Enable a GPU runtime in Colab."
)

OUTPUT = BASE / "complete_evaluation_report"
OUTPUT.mkdir(parents=True, exist_ok=True)

model = YOLO(str(BEST))

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

root = Path(cfg.get("path", DATA_YAML.parent))
if not root.is_absolute():
    root = DATA_YAML.parent / root

print("Model:", BEST)
print("Dataset:", DATA_YAML)
print("GPU:", torch.cuda.get_device_name(0))


# -------------------- 2. TRAINING HISTORY --------------------

history_path = BEST.parent.parent / "results.csv"
history_summary = {}

if history_path.exists():
    history = pd.read_csv(history_path)
    history.columns = history.columns.str.strip()

    if "epoch" in history.columns:
        history["epoch"] = pd.to_numeric(
            history["epoch"], errors="coerce"
        )

    # Best epoch according to validation mask mAP@50
    map50_col = "metrics/mAP50(M)"

    if map50_col in history.columns:
        valid = history.dropna(subset=[map50_col])

        if not valid.empty:
            best_row = valid.loc[valid[map50_col].idxmax()]
            history_summary["Best validation epoch"] = (
                int(best_row["epoch"]) + 1
                if pd.notna(best_row.get("epoch"))
                else "Not available"
            )
            history_summary["Best validation mask mAP50"] = float(
                best_row[map50_col]
            )

    # Save the complete epoch-by-epoch training history
    history.to_csv(OUTPUT / "training_history.csv", index=False)

    print("\nTraining history found:", history_path)
else:
    history = None
    print("\nTraining results.csv not found; skipping history summary.")


# -------------------- 3. RESOLVE SPLIT IMAGE DIRECTORIES --------------------

def get_image_dirs(split):
    entry = cfg.get(split)

    if entry is None:
        raise ValueError(
            f"The dataset YAML has no '{split}' entry."
        )

    entries = entry if isinstance(entry, list) else [entry]
    dirs = []

    for item in entries:
        p = Path(item)
        if not p.is_absolute():
            p = root / p

        if p.is_dir():
            dirs.append(p)
        elif p.is_file() and p.suffix.lower() == ".txt":
            raise ValueError(
                f"{split} uses an image-list text file ({p}). "
                "This report cell expects split directories."
            )
        else:
            raise FileNotFoundError(
                f"Could not find the {split} path: {p}"
            )

    return dirs


EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def get_images(split):
    images = []
    for directory in get_image_dirs(split):
        images.extend(
            p for p in directory.rglob("*")
            if p.is_file() and p.suffix.lower() in EXTENSIONS
        )
    return sorted(set(images))


split_images = {
    split: get_images(split)
    for split in ("train", "val", "test")
}

for split, images in split_images.items():
    print(f"{split.upper()} images: {len(images)}")
    if not images:
        raise RuntimeError(f"No images found for split: {split}")


# -------------------- 4. EVALUATE TRAIN / VALIDATION / TEST --------------------

all_metrics = {}

for split in ("train", "val", "test"):
    print(f"\n{'=' * 25} {split.upper()} EVALUATION {'=' * 25}")

    result = model.val(
        data=str(DATA_YAML),
        split=split,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0,
        workers=2,
        plots=False,
        verbose=False,
        project=str(OUTPUT),
        name=f"{split}_evaluation",
        exist_ok=True
    )

    values = result.results_dict
    all_metrics[split] = {
        key: float(value)
        for key, value in values.items()
        if isinstance(value, (int, float, np.number))
    }

    for key, value in all_metrics[split].items():
        print(f"{key}: {value:.4f}")


# -------------------- 5. PIXEL-LEVEL DICE / IoU ON TEST SET --------------------

# Ground-truth YOLO polygon labels are rasterized into pixel masks.
# Predicted segmentation polygons are rasterized at original image size.
# Scores are aggregated across test images separately for each class.

def label_dir_for_image_dir(image_dir):
    parts = list(image_dir.parts)

    image_indices = [
        i for i, part in enumerate(parts) if part == "images"
    ]

    if not image_indices:
        raise ValueError(
            f"Cannot infer labels directory from {image_dir}. "
            "Expected a directory path containing an 'images' folder."
        )

    idx = image_indices[-1]
    label_parts = parts[:idx] + ["labels"] + parts[idx + 1:]
    return Path(*label_parts)


def ground_truth_masks(label_path, width, height):
    masks = {
        class_id: np.zeros((height, width), dtype=np.uint8)
        for class_id in CLASS_NAMES
    }

    if not label_path.exists():
        # A missing label file in YOLO format means no annotated objects.
        return masks

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        values = line.strip().split()
        if len(values) < 7:
            continue

        class_id = int(float(values[0]))
        if class_id not in masks:
            continue

        coords = np.asarray(
            [float(v) for v in values[1:]],
            dtype=np.float32
        )

        if len(coords) < 6 or len(coords) % 2 != 0:
            continue

        points = coords.reshape(-1, 2)
        points[:, 0] *= width
        points[:, 1] *= height
        points[:, 0] = np.clip(points[:, 0], 0, width - 1)
        points[:, 1] = np.clip(points[:, 1], 0, height - 1)

        cv2.fillPoly(
            masks[class_id],
            [np.round(points).astype(np.int32)],
            1
        )

    return masks


# Map every image to its corresponding labels directory.
label_roots = {}
for image_dir in get_image_dirs("test"):
    label_roots[image_dir] = label_dir_for_image_dir(image_dir)


def find_label_path(image_path):
    for image_dir, label_dir in label_roots.items():
        try:
            relative = image_path.relative_to(image_dir)
            return label_dir / relative.with_suffix(".txt")
        except ValueError:
            continue

    raise ValueError(f"Cannot map image to label path: {image_path}")


counts = {
    class_id: {"intersection": 0, "pred": 0, "gt": 0}
    for class_id in CLASS_NAMES
}

# Batch inference; masks are converted to original image coordinates.
predictions = model.predict(
    source=[str(p) for p in split_images["test"]],
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    conf=CONF_THRESHOLD,
    device=0,
    retina_masks=True,
    stream=True,
    verbose=False
)

processed = 0

for result in predictions:
    image_path = Path(result.path)
    height, width = result.orig_shape

    gt_masks = ground_truth_masks(
        find_label_path(image_path), width, height
    )

    pred_masks = {
        class_id: np.zeros((height, width), dtype=np.uint8)
        for class_id in CLASS_NAMES
    }

    if result.masks is not None and result.boxes is not None:
        pred_classes = (
            result.boxes.cls.detach().cpu().numpy().astype(int)
        )

        for polygon, class_id in zip(
            result.masks.xy, pred_classes
        ):
            if class_id not in pred_masks or len(polygon) < 3:
                continue

            points = np.round(polygon).astype(np.int32)
            points[:, 0] = np.clip(points[:, 0], 0, width - 1)
            points[:, 1] = np.clip(points[:, 1], 0, height - 1)

            cv2.fillPoly(
                pred_masks[class_id], [points], 1
            )

    for class_id in CLASS_NAMES:
        pred = pred_masks[class_id].astype(bool)
        gt = gt_masks[class_id].astype(bool)

        counts[class_id]["intersection"] += int(
            np.logical_and(pred, gt).sum()
        )
        counts[class_id]["pred"] += int(pred.sum())
        counts[class_id]["gt"] += int(gt.sum())

    processed += 1
    if processed % 50 == 0:
        print(f"Pixel metrics processed: {processed}/{len(split_images['test'])}")

dice_iou = {}

for class_id, name in CLASS_NAMES.items():
    c = counts[class_id]
    intersection = c["intersection"]
    denominator = c["pred"] + c["gt"]
    union = denominator - intersection

    dice = (2 * intersection / denominator) if denominator else 1.0
    iou = (intersection / union) if union else 1.0

    dice_iou[name] = {
        "Dice": dice,
        "Pixel_IoU": iou,
        "GT_pixels": c["gt"],
        "Predicted_pixels": c["pred"]
    }

# Overall foreground metrics, combining the two classes' pixel counts.
total_intersection = sum(c["intersection"] for c in counts.values())
total_pred = sum(c["pred"] for c in counts.values())
total_gt = sum(c["gt"] for c in counts.values())
total_union = total_pred + total_gt - total_intersection

dice_iou["Overall_Foreground"] = {
    "Dice": (
        2 * total_intersection / (total_pred + total_gt)
        if total_pred + total_gt else 1.0
    ),
    "Pixel_IoU": (
        total_intersection / total_union if total_union else 1.0
    ),
    "GT_pixels": total_gt,
    "Predicted_pixels": total_pred
}

print("\n========== TEST PIXEL-LEVEL DICE / IoU ==========")
for name, metrics in dice_iou.items():
    print(
        f"{name}: Dice={metrics['Dice']:.4f}, "
        f"Pixel IoU={metrics['Pixel_IoU']:.4f}"
    )


# -------------------- 6. SAVE CSV FILES --------------------

metrics_df = pd.DataFrame(all_metrics).T
metrics_df.index.name = "Dataset_Split"
metrics_df.to_csv(OUTPUT / "train_validation_test_metrics.csv")

dice_df = pd.DataFrame(dice_iou).T
dice_df.index.name = "Class"
dice_df.to_csv(OUTPUT / "test_dice_iou_by_class.csv")


# -------------------- 7. CREATE HUMAN-READABLE REPORT --------------------

report_path = OUTPUT / "CDAC_YOLO26_Segmentation_Report.txt"

with open(report_path, "w") as f:
    f.write("CDAC AGRICULTURE SEGMENTATION EVALUATION REPORT\n")
    f.write("=" * 55 + "\n\n")
    f.write("Model: YOLO26 Nano Segmentation\n")
    f.write(f"Checkpoint: {BEST}\n")
    f.write(f"Image size: {IMG_SIZE}\n")
    f.write(f"Evaluation batch size: {BATCH_SIZE}\n")
    f.write(f"Pixel metric confidence threshold: {CONF_THRESHOLD}\n\n")

    f.write("DATASET SIZES\n")
    for split, images in split_images.items():
        f.write(f"{split.title()}: {len(images)} images\n")

    f.write("\nTRAIN / VALIDATION / TEST METRICS\n")
    f.write("-" * 55 + "\n")

    for split, metrics in all_metrics.items():
        f.write(f"\n{split.upper()}\n")
        for key, value in metrics.items():
            f.write(f"{key}: {value:.6f}\n")

    f.write("\nTRAINING HISTORY\n")
    f.write("-" * 55 + "\n")
    if history_summary:
        for key, value in history_summary.items():
            f.write(f"{key}: {value}\n")
    else:
        f.write("Training history summary unavailable.\n")

    f.write("\nTEST PIXEL-LEVEL DICE AND IOU\n")
    f.write("-" * 55 + "\n")
    for name, metrics in dice_iou.items():
        f.write(
            f"{name}: Dice={metrics['Dice']:.6f}, "
            f"Pixel IoU={metrics['Pixel_IoU']:.6f}, "
            f"GT pixels={metrics['GT_pixels']}, "
            f"Predicted pixels={metrics['Predicted_pixels']}\n"
        )

    f.write("\nINTERPRETATION NOTES\n")
    f.write(
        "- Precision, recall and mAP are object-level detection/segmentation "
        "metrics from Ultralytics.\n"
    )
    f.write(
        "- Dice and Pixel IoU here are pixel-overlap metrics on test images, "
        "using confidence threshold 0.25.\n"
    )
    f.write(
        "- Training-set evaluation measures performance on data used for "
        "learning; it is not an independent generalization estimate.\n"
    )
    f.write(
        "- The test set should be reserved for final evaluation and not used "
        "repeatedly to tune the model.\n"
    )

print("\n" + "=" * 55)
print("EVALUATION REPORT COMPLETED")
print("Text report:", report_path)
print("Split metrics CSV:", OUTPUT / "train_validation_test_metrics.csv")
print("Test Dice/IoU CSV:", OUTPUT / "test_dice_iou_by_class.csv")
if history is not None:
    print("Training history CSV:", OUTPUT / "training_history.csv")
print("=" * 55)
